In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

In [2]:
from src.rarekg.curated import downloaders

API_KEY = os.getenv("UMLS_API_KEY")


In [3]:
downloaders.download_rxnorm_full(API_KEY)

In [4]:
downloaders.ensure_download_directories()
downloaders.download_orphanet_alignment()
downloaders.download_orphanet_classifications()
downloaders.download_orphanet_genes()
downloaders.download_orphanet_phenotypes()

downloaders.download_hpo()
downloaders.download_hgnc_complete()

downloaders.download_snomed_ct_international(API_KEY)

Extracted Orphanet alignment files.
Extracted Orphanet alignment files.
Found 68 classification files.
Found 68 classification files.


In [3]:
downloaders.download_snomed_ct_us(API_KEY)

In [4]:
import pandas as pd

DESC_FILE = "../data/raw/snomed_ct/SnomedCT_InternationalRF2_PRODUCTION_20251001T120000Z/Snapshot/Terminology/sct2_Description_Snapshot-en_INT_20251001.txt"

df = pd.read_csv(DESC_FILE, sep="\t", dtype=str)
df = df[(df["active"]=="1") & (df["typeId"]=="900000000000013009")]  # Synonyms only

def find(term):
    m = df[df["term"].str.contains(term, case=False, na=False)]
    return m[["conceptId","term"]].drop_duplicates().head(50)

print("HSCT hits:\n", find("stem cell transplant"))
print("ERT hits:\n", find("enzyme replacement therapy"))  # likely empty in INT
print("enzyme hits:\n", find("enzyme replacement therapy"))                  # may reveal specific procedures


HSCT hits:
                   conceptId                                               term
334783            234336002                   Hemopoietic stem cell transplant
334784            234336002                  Haemopoietic stem cell transplant
926332            397554006                   Limbal stem cell transplantation
980090            278257006         Peripheral blood stem cell transplantation
1056203           425983008   Autologous peripheral blood stem cell transplant
1056204           425843001   Allogeneic peripheral blood stem cell transplant
1200357           698075004  Syngeneic peripheral blood stem cell transplan...
1229076     153351000119102         History of peripheral stem cell transplant
1538832          1172516002  HDCT-SCT - high-dose chemotherapy with stem ce...
1538833          1172516002   High-dose chemotherapy with stem cell transplant
1594560          1269349006                               Stem cell transplant
1607009  232417611000119100  Thrombotic 

In [3]:
downloaders.download_umls_mrconso(API_KEY)

(PosixPath('/home/guests/andreea_magureanu/rare_disease_pipeline/data/raw/umls-2025AB-mrconso.zip'),
 PosixPath('/home/guests/andreea_magureanu/rare_disease_pipeline/data/raw/MRCONSO.RRF'))

In [4]:
import pandas as pd
from pathlib import Path

RRF = Path("../data/raw/MRCONSO.RRF")  # from the MRCONSO download
# MRCONSO is pipe-delimited with no header; keep only columns we need
cols = ["CUI","LAT","TS","LUI","STT","SUI","ISPREF","AUI","SAUI","SCUI","SDUI",
        "SAB","TTY","CODE","STR","SRL","SUPPRESS","CVF"]
df = pd.read_csv(RRF, sep="|", header=None, dtype=str, usecols=range(len(cols)),
                 names=cols, quoting=3, encoding="utf-8")

# 1) Find a CUI by the exact phrase (case-insensitive)
q = df[df["STR"].str.contains(r"\benzyme replacement therapy\b", case=False, na=False)]
print(q[["CUI","SAB","TTY","CODE","STR"]].drop_duplicates().head(20))

# 2) Given a CUI, list all terms across sources (including SNOMED if present)
if not q.empty:
    cui = q.iloc[0]["CUI"]
    all_terms = df[df["CUI"] == cui][["SAB","TTY","CODE","STR"]].drop_duplicates()
    print("\nAll terms for CUI", cui)
    print(all_terms.head(50))


               CUI   SAB   TTY        CODE  \
6009553   C0598391   NCI    SY      C16221   
6009554   C0598391   MSH    MH     D056947   
6009555   C0598391   CSP    ET   1042-1841   
6009556   C0598391   CHV    PT  0000041885   
10610017  C1854836  OMIM  PTCS  MTHU011230   
15190840  C4523842   MDR   LLT    10079612   
15190841  C4523842   MDR    PT    10079612   

                                                        STR  
6009553                          Enzyme Replacement Therapy  
6009554                          Enzyme Replacement Therapy  
6009555                          enzyme replacement therapy  
6009556                          enzyme replacement therapy  
10610017  Enzyme replacement therapy has not been effective  
15190840              Pancreatic enzyme replacement therapy  
15190841              Pancreatic enzyme replacement therapy  

All terms for CUI C0598391
            SAB TTY        CODE                                      STR
6009550  MSHCZE  MH     D056947   